# Iterative Refinement Wrapper: LLaDA Denoising + Noise-Injection Training on ModernBERT

This notebook implements an **iterative refinement wrapper** around `dllm-hub/ModernBERT-base-chat-v0.1` combining two complementary techniques:

| Component | Role | Mechanism |
|-----------|------|----------|
| **LLaDA Denoising** | Inference-time iterative refinement | Masked diffusion: fully masked → progressively unmasked via confidence-guided remasking |
| **Noise-Injection Training** | Lightweight diffusion-flavor fine-tuning | Corrupt response tokens with variable-rate masking (MDLM SFT loss) |

## References
- **dLLM** ([arXiv:2602.22661](https://arxiv.org/abs/2602.22661)) – unified DLM framework; BERT-Chat recipe
- **LLaDA** ([arXiv:2502.09992](https://arxiv.org/abs/2502.09992)) – masked diffusion LM at 8B scale
- [`dllm-hub/ModernBERT-base-chat-v0.1`](https://huggingface.co/dllm-hub/ModernBERT-base-chat-v0.1)
- [`ZHZisZZ/dllm`](https://github.com/ZHZisZZ/dllm) – reference implementation
- [`ML-GSAI/LLaDA`](https://github.com/ML-GSAI/LLaDA) – original LLaDA code

---
## Architecture Overview

```
  ┌─────────────────────────────────────────────────────────────────────┐
  │                    ITERATIVE REFINEMENT WRAPPER                     │
  │                                                                     │
  │  NOISE-INJECTION TRAINING (fine-tune)   LLADA DENOISING (inference) │
  │  ┌─────────────────────────┐            ┌─────────────────────────┐ │
  │  │  Clean response r₀      │            │  t=1: [MASK][MASK]...   │ │
  │  │         ↓               │            │         ↓               │ │
  │  │  Corrupt at rate t~U[0,1]│            │  Forward pass → logits  │ │
  │  │  r_t = mask(r₀, t)     │            │         ↓               │ │
  │  │         ↓               │            │  Unmask top-k confident  │ │
  │  │  ModernBERT(r_t | prompt)│            │         ↓               │ │
  │  │         ↓               │            │  Remask low-confidence  │ │
  │  │  MDLM Loss: -log p(r₀)  │            │         ↓               │ │
  │  │  only on masked positions│            │  t=0: final output      │ │
  │  └─────────────────────────┘            └─────────────────────────┘ │
  └─────────────────────────────────────────────────────────────────────┘
```

## 1. Installation

In [ ]:
# Install core dependencies
!pip install -q transformers==4.47.0 torch datasets tqdm numpy

# Optionally install dllm for its full training pipeline
# !git clone https://github.com/ZHZisZZ/dllm && pip install -e dllm/

import importlib, subprocess, sys

def check_import(pkg):
    try: importlib.import_module(pkg); print(f'  ✓ {pkg}')
    except ImportError: print(f'  ✗ {pkg} not found')

for p in ['torch', 'transformers', 'datasets', 'numpy']:
    check_import(p)

## 2. Imports & Device Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple
from tqdm.auto import tqdm, trange
from transformers import AutoTokenizer, AutoModelForMaskedLM
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. Load ModernBERT-base-chat-v0.1

In [ ]:
MODEL_ID = 'dllm-hub/ModernBERT-base-chat-v0.1'

print(f'Loading tokenizer from {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f'Loading model ...')
model = AutoModelForMaskedLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if DEVICE.type == 'cuda' else torch.float32,
)
model = model.to(DEVICE)
model.eval()

# Inspect mask token
MASK_ID = tokenizer.mask_token_id
EOS_ID  = tokenizer.eos_token_id
PAD_ID  = tokenizer.pad_token_id
print(f'\nTokenizer special tokens:')
print(f'  MASK id : {MASK_ID}  ({tokenizer.mask_token})')
print(f'  EOS id  : {EOS_ID}')
print(f'  PAD id  : {PAD_ID}')
print(f'\nModel params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

## 4. Core Primitives: Noise Schedule & Masking

LLaDA uses a **linear absorption schedule** $\alpha_t = 1 - t$ where $t \in [0,1]$:
- At $t=0$: all tokens visible (clean)
- At $t=1$: all tokens masked (fully corrupted)

The forward (noising) process independently masks each token with probability $t$.

In [ ]:
@dataclass
class DiffusionConfig:
    """Configuration for the iterative refinement wrapper."""
    # ── Inference ──────────────────────────────────────────────────────
    num_steps: int   = 64       # denoising steps (more = higher quality, slower)
    gen_length: int  = 128      # number of tokens to generate
    temperature: float = 0.0    # Gumbel noise temperature (0 = greedy)
    remask_strategy: str = 'confidence'  # 'random' | 'confidence'
    # ── Training ───────────────────────────────────────────────────────
    mask_rate_min: float = 0.1  # minimum masking rate for noise injection
    mask_rate_max: float = 1.0  # maximum masking rate
    label_smoothing: float = 0.0


# ─── Gumbel noise (used for stochastic sampling) ───────────────────────────
def add_gumbel_noise(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    """Add Gumbel noise for stochastic token selection (temperature=0 → greedy)."""
    if temperature == 0:
        return logits
    logits = logits.to(torch.float64)
    noise = torch.rand_like(logits, dtype=torch.float64)
    gumbel = (-torch.log(noise)).pow(temperature)          # Gumbel(0,1)
    return logits.exp() / gumbel                           # Gumbel-max trick


# ─── Linear noise schedule ─────────────────────────────────────────────────
def alpha_t(t: float) -> float:
    """Proportion of tokens that remain UNMASKED at diffusion time t."""
    return 1.0 - t                                         # LLaDA linear schedule


# ─── Forward process: add noise to a clean token sequence ─────────────────
def forward_noise(
    input_ids: torch.Tensor,
    t: float,
    mask_id: int,
    response_mask: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Apply forward (noising) process:
      - mask each token independently with probability t
      - if response_mask is given, only mask response positions (SFT mode)

    Args:
        input_ids     : (B, L) clean token ids
        t             : masking ratio in [0, 1]
        mask_id       : tokenizer's [MASK] token id
        response_mask : (B, L) bool tensor; True = maskable position

    Returns:
        noisy_ids     : (B, L) corrupted input
        is_masked     : (B, L) bool, True where tokens were masked
    """
    B, L = input_ids.shape
    noise_prob = torch.full((B, L), t, device=input_ids.device)

    if response_mask is not None:
        # Only allow masking of response tokens (SFT convention in LLaDA/dLLM)
        noise_prob = noise_prob * response_mask.float()

    is_masked = torch.bernoulli(noise_prob).bool()
    noisy_ids = input_ids.clone()
    noisy_ids[is_masked] = mask_id
    return noisy_ids, is_masked


# ─── Token-transfer schedule for iterative denoising ──────────────────────
def get_transfer_schedule(mask_index: torch.Tensor, steps: int) -> torch.Tensor:
    """
    Compute how many masked tokens to reveal at each denoising step.
    Distributes the total number of masks as evenly as possible across steps.

    Returns:
        num_transfer : (B, steps) int64 tensor
    """
    B = mask_index.shape[0]
    mask_count = mask_index.sum(dim=1, keepdim=True)       # (B, 1)
    base       = mask_count // steps
    remainder  = mask_count % steps
    schedule   = base.expand(B, steps).clone()
    for i in range(B):
        schedule[i, :remainder[i].item()] += 1
    return schedule                                        # (B, steps)


print('Primitives defined.')
# Quick sanity check
dummy = torch.tensor([[1, 2, 3, 4, 5]])
noisy, masked = forward_noise(dummy, t=0.6, mask_id=MASK_ID)
print(f'  Original : {dummy[0].tolist()}')
print(f'  Noisy    : {noisy[0].tolist()}  (t=0.6, ~3 tokens masked)')

## 5. LLaDA Iterative Denoising (Inference)

The reverse process proceeds from $t=1$ (fully masked) to $t=0$ (clean) over `num_steps` steps.

At each step:
1. Run a forward pass to get per-token logits
2. Sample candidate tokens (optionally with Gumbel noise)
3. Reveal the `k` most-confident masked positions
4. (Optional) remask the least-confident previously-revealed positions

In [ ]:
@torch.no_grad()
def llada_generate(
    model,
    prompt_ids: torch.Tensor,
    cfg: DiffusionConfig,
    mask_id: int,
    eos_id: Optional[int] = None,
    verbose: bool = False,
) -> torch.Tensor:
    """
    LLaDA-style iterative denoising generation.

    Args:
        model       : HF MaskedLM model
        prompt_ids  : (1, P) prompt token ids (prompt tokens stay fixed)
        cfg         : DiffusionConfig
        mask_id     : tokenizer mask token id
        eos_id      : tokenizer eos token id (used to suppress post-EOS tokens)
        verbose     : print per-step mask count

    Returns:
        generated   : (1, gen_length) generated token ids
    """
    model.eval()
    B = prompt_ids.shape[0]
    P = prompt_ids.shape[1]
    L = cfg.gen_length

    # ── Initialise: full sequence = [prompt | MASK * gen_length] ─────────
    response_ids = torch.full((B, L), mask_id, dtype=torch.long, device=DEVICE)
    input_ids    = torch.cat([prompt_ids.to(DEVICE), response_ids], dim=1)  # (B, P+L)

    # Mask is only over the response suffix
    mask_index   = torch.zeros(B, P + L, dtype=torch.bool, device=DEVICE)
    mask_index[:, P:] = True

    # ── Token-transfer schedule ───────────────────────────────────────────
    transfer_schedule = get_transfer_schedule(mask_index, cfg.num_steps)  # (B, steps)

    if verbose:
        print(f'Generating {L} tokens in {cfg.num_steps} steps ...')

    for step in range(cfg.num_steps):
        # ── Forward pass ─────────────────────────────────────────────────
        logits = model(input_ids).logits                   # (B, P+L, V)

        # ── Sample candidates from all masked positions ───────────────────
        if cfg.temperature > 0:
            sampled = add_gumbel_noise(logits, cfg.temperature).argmax(dim=-1)  # (B, P+L)
        else:
            sampled = logits.argmax(dim=-1)

        # ── Confidence = max softmax prob at each position ────────────────
        confidence = logits.softmax(dim=-1).max(dim=-1).values  # (B, P+L)

        # Set confidence of already-unmasked positions to -inf so we skip them
        confidence[~mask_index] = -torch.inf

        # ── How many tokens to reveal at this step ────────────────────────
        k = transfer_schedule[:, step]                     # (B,)

        for b in range(B):
            ki = k[b].item()
            if ki == 0:
                continue

            if cfg.remask_strategy == 'confidence':
                # Reveal the ki most-confident masked positions
                masked_positions = mask_index[b].nonzero(as_tuple=True)[0]
                if len(masked_positions) == 0:
                    continue
                conf_at_masked = confidence[b][masked_positions]
                topk_idx       = conf_at_masked.topk(min(ki, len(masked_positions))).indices
                reveal_pos     = masked_positions[topk_idx]
            else:
                # Random strategy: pick ki random masked positions
                masked_positions = mask_index[b].nonzero(as_tuple=True)[0]
                perm             = torch.randperm(len(masked_positions), device=DEVICE)
                reveal_pos       = masked_positions[perm[:ki]]

            input_ids[b, reveal_pos]    = sampled[b, reveal_pos]
            mask_index[b, reveal_pos]   = False

        if verbose:
            remaining = mask_index[:, P:].sum().item()
            print(f'  step {step+1:3d}/{cfg.num_steps}  masks remaining: {remaining}')

    generated = input_ids[:, P:]                           # strip prompt
    return generated


print('llada_generate defined.')

## 6. Prompt Formatting & Inference Demo

ModernBERT-chat-v0.1 was fine-tuned with a specific chat template. We use the tokenizer's built-in `apply_chat_template`.

In [ ]:
def format_prompt(tokenizer, user_message: str) -> torch.Tensor:
    """Format a single-turn user message using the model's chat template."""
    messages = [{'role': 'user', 'content': user_message}]
    try:
        # apply_chat_template adds BOS/EOS, formats roles, etc.
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        # Fallback: manual format
        prompt_text = f'<|user|>\n{user_message}\n<|assistant|>\n'

    ids = tokenizer(prompt_text, return_tensors='pt').input_ids
    return ids


def decode_response(tokenizer, generated_ids: torch.Tensor) -> str:
    """Decode generated ids, stopping at EOS."""
    tokens = generated_ids[0].tolist()
    eos    = tokenizer.eos_token_id
    if eos and eos in tokens:
        tokens = tokens[:tokens.index(eos)]
    # Also strip mask tokens (shouldn't remain, but just in case)
    tokens = [t for t in tokens if t != tokenizer.mask_token_id]
    return tokenizer.decode(tokens, skip_special_tokens=True).strip()


# ── Quick demo ────────────────────────────────────────────────────────────
cfg_demo = DiffusionConfig(
    num_steps   = 64,
    gen_length  = 128,
    temperature = 0.0,
    remask_strategy = 'confidence',
)

DEMO_PROMPT = 'Explain the difference between a diffusion model and an autoregressive model in two sentences.'

prompt_ids = format_prompt(tokenizer, DEMO_PROMPT)
print(f'Prompt ({prompt_ids.shape[1]} tokens):')
print(f'  "{DEMO_PROMPT}"\n')

print('Running LLaDA iterative denoising ...')
gen_ids  = llada_generate(model, prompt_ids, cfg_demo, MASK_ID, EOS_ID, verbose=False)
response = decode_response(tokenizer, gen_ids)

print('\n── Generated Response ──────────────────────────────────────────────')
print(response)
print('────────────────────────────────────────────────────────────────────')

## 7. Noise-Injection Training (Diffusion SFT)

We fine-tune with the **MDLM SFT loss** from LLaDA §2.3 / dLLM:

$$\mathcal{L}_{\text{MDLM}} = \mathbb{E}_{t \sim U[0,1],\, x_t \sim q(x_t | x_0)} \left[ \frac{1}{|\text{masked}|} \sum_{i \in \text{masked}} -\log p_\theta(x_0^i | x_t) \right]$$

Key design choices:
- **Variable masking rate** `t ~ U[mask_rate_min, mask_rate_max]` per batch  
- **Response-only masking**: prompt tokens are never corrupted (standard SFT convention)  
- Loss averaged only over masked positions (avoids dilution from clean tokens)

In [ ]:
# ── MDLM SFT Loss ────────────────────────────────────────────────────────

def mdlm_sft_loss(
    model,
    input_ids: torch.Tensor,
    response_mask: torch.Tensor,
    cfg: DiffusionConfig,
    mask_id: int,
) -> torch.Tensor:
    """
    Compute MDLM SFT (noise-injection) loss.

    Args:
        input_ids     : (B, L) clean token ids (prompt + response concatenated)
        response_mask : (B, L) bool, True = response positions (maskable)
        cfg           : DiffusionConfig
        mask_id       : tokenizer mask id

    Returns:
        loss : scalar tensor
    """
    B, L = input_ids.shape

    # Sample one masking rate per example in [mask_rate_min, mask_rate_max]
    t_vals = torch.empty(B, device=input_ids.device).uniform_(
        cfg.mask_rate_min, cfg.mask_rate_max
    )                                                      # (B,)

    # Build noisy inputs using per-example t
    noisy_ids   = input_ids.clone()
    is_masked   = torch.zeros(B, L, dtype=torch.bool, device=input_ids.device)

    for b in range(B):
        t = t_vals[b].item()
        # Only mask positions in the response
        resp_pos = response_mask[b].nonzero(as_tuple=True)[0]
        if len(resp_pos) == 0:
            continue
        mask_flags = torch.bernoulli(torch.full((len(resp_pos),), t,
                                                device=input_ids.device)).bool()
        masked_resp_pos        = resp_pos[mask_flags]
        noisy_ids[b, masked_resp_pos] = mask_id
        is_masked[b, masked_resp_pos] = True

    # Forward pass through ModernBERT
    logits = model(noisy_ids).logits                       # (B, L, V)

    # NLL loss only at masked positions
    if is_masked.sum() == 0:
        return torch.tensor(0.0, requires_grad=True, device=input_ids.device)

    loss = F.cross_entropy(
        logits[is_masked],                                 # (N, V)  N = total masked
        input_ids[is_masked],                              # (N,)    ground truth
        label_smoothing=cfg.label_smoothing,
    )
    return loss


print('mdlm_sft_loss defined.')

## 8. Dataset Wrapper

A minimal `Dataset` that tokenises `(prompt, response)` pairs and produces the `response_mask` needed for SFT.

In [ ]:
class DiffusionSFTDataset(Dataset):
    """
    Tokenises (prompt, response) pairs for MDLM SFT training.

    Each item returns:
        input_ids     : (max_length,)  prompt + response, padded
        response_mask : (max_length,)  True = response position
        attention_mask: (max_length,)  False = padding
    """

    def __init__(
        self,
        examples: List[Dict[str, str]],   # list of {'prompt': ..., 'response': ...}
        tokenizer,
        max_length: int = 256,
    ):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.items      = []

        for ex in examples:
            # Format prompt using chat template
            messages = [{'role': 'user', 'content': ex['prompt']}]
            try:
                prompt_text = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
            except Exception:
                prompt_text = f"<|user|>\n{ex['prompt']}\n<|assistant|>\n"

            prompt_enc   = tokenizer(prompt_text, add_special_tokens=False)
            response_enc = tokenizer(
                ex['response'] + tokenizer.eos_token,
                add_special_tokens=False,
            )

            p_ids = prompt_enc['input_ids']
            r_ids = response_enc['input_ids']

            # Truncate to fit max_length
            total   = len(p_ids) + len(r_ids)
            if total > max_length:
                r_ids = r_ids[:max(1, max_length - len(p_ids))]

            combined = p_ids + r_ids
            pad_len  = max_length - len(combined)
            pad_id   = tokenizer.pad_token_id or 0

            input_ids      = combined + [pad_id] * pad_len
            response_mask  = [False] * len(p_ids) + [True] * len(r_ids) + [False] * pad_len
            attention_mask = [1] * len(combined) + [0] * pad_len

            self.items.append({
                'input_ids'     : torch.tensor(input_ids,      dtype=torch.long),
                'response_mask' : torch.tensor(response_mask,  dtype=torch.bool),
                'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            })

    def __len__(self):  return len(self.items)
    def __getitem__(self, idx): return self.items[idx]


# ── Tiny synthetic dataset for demonstration ──────────────────────────────
DEMO_DATA = [
    {'prompt': 'What is a language model?',
     'response': 'A language model is a probability distribution over sequences of tokens, '
                 'trained to predict likely continuations of text.'},
    {'prompt': 'Explain masked diffusion in one sentence.',
     'response': 'Masked diffusion corrupts input tokens by replacing them with [MASK] and '
                 'trains the model to reconstruct them using bidirectional context.'},
    {'prompt': 'What is ModernBERT?',
     'response': 'ModernBERT is an improved BERT-style encoder with flash attention, '
                 'rotary position embeddings, and a longer context window.'},
    {'prompt': 'How does LLaDA generate text?',
     'response': 'LLaDA starts with a fully masked response and iteratively unmasks tokens '
                 'in order of model confidence over multiple denoising steps.'},
    {'prompt': 'What is the forward diffusion process?',
     'response': 'The forward process gradually corrupts data by adding noise; '
                 'for discrete text, this means progressively masking tokens.'},
]

dataset    = DiffusionSFTDataset(DEMO_DATA, tokenizer, max_length=256)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

batch = next(iter(dataloader))
print(f'Batch shapes:')
print(f'  input_ids     : {batch["input_ids"].shape}')
print(f'  response_mask : {batch["response_mask"].shape}')
print(f'  response tokens per example: {batch["response_mask"].sum(dim=1).tolist()}')

## 9. Iterative Refinement Wrapper Class

This is the main wrapper that glues together:
- **Training**: `noise_injection_train()` — MDLM SFT fine-tuning
- **Inference**: `generate()` — LLaDA iterative denoising
- **Logging**: loss curves, per-step mask counts

In [ ]:
class IterativeRefinementWrapper:
    """
    Combines LLaDA denoising inference and noise-injection (MDLM SFT) training
    on top of a HuggingFace AutoModelForMaskedLM (e.g. ModernBERT-base-chat-v0.1).

    Usage::

        wrapper = IterativeRefinementWrapper(model, tokenizer, cfg)
        wrapper.noise_injection_train(dataloader, num_epochs=3)
        response = wrapper.generate("Tell me about diffusion models.")
    """

    def __init__(
        self,
        model,
        tokenizer,
        cfg: DiffusionConfig,
        device: torch.device = DEVICE,
    ):
        self.model     = model
        self.tokenizer = tokenizer
        self.cfg       = cfg
        self.device    = device
        self.mask_id   = tokenizer.mask_token_id
        self.eos_id    = tokenizer.eos_token_id
        self.loss_log: List[float] = []

    # ── Training ──────────────────────────────────────────────────────────
    def noise_injection_train(
        self,
        dataloader: DataLoader,
        num_epochs: int = 3,
        lr: float = 2e-5,
        grad_clip: float = 1.0,
        warmup_steps: int = 10,
        log_every: int = 1,
    ) -> List[float]:
        """
        Fine-tune using MDLM SFT (noise-injection) loss.

        Args:
            dataloader  : yields batches with 'input_ids' and 'response_mask'
            num_epochs  : number of training epochs
            lr          : learning rate (AdamW)
            grad_clip   : gradient norm clipping
            warmup_steps: linear LR warmup
            log_every   : print loss every N batches

        Returns:
            loss_log    : list of per-step losses
        """
        self.model.train()
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=0.01)

        total_steps = num_epochs * len(dataloader)
        scheduler   = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=lr, total_steps=total_steps,
            pct_start=warmup_steps / max(total_steps, warmup_steps + 1),
        )

        global_step = 0
        for epoch in range(num_epochs):
            epoch_losses = []
            pbar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{num_epochs}', leave=False)

            for batch in pbar:
                input_ids      = batch['input_ids'].to(self.device)
                response_mask  = batch['response_mask'].to(self.device)

                optimizer.zero_grad()
                loss = mdlm_sft_loss(
                    self.model, input_ids, response_mask,
                    self.cfg, self.mask_id,
                )
                loss.backward()

                if grad_clip > 0:
                    nn.utils.clip_grad_norm_(self.model.parameters(), grad_clip)

                optimizer.step()
                scheduler.step()

                lv = loss.item()
                epoch_losses.append(lv)
                self.loss_log.append(lv)
                global_step += 1

                if global_step % log_every == 0:
                    pbar.set_postfix({'loss': f'{lv:.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})

            avg = np.mean(epoch_losses)
            print(f'  Epoch {epoch+1}: avg loss = {avg:.4f}')

        self.model.eval()
        return self.loss_log

    # ── Inference ─────────────────────────────────────────────────────────
    def generate(
        self,
        user_message: str,
        verbose: bool = False,
    ) -> str:
        """
        Generate a response using LLaDA iterative denoising.

        Args:
            user_message : plain text query
            verbose      : print per-step stats

        Returns:
            decoded response string
        """
        prompt_ids = format_prompt(self.tokenizer, user_message)
        gen_ids    = llada_generate(
            self.model, prompt_ids, self.cfg,
            self.mask_id, self.eos_id, verbose=verbose,
        )
        return decode_response(self.tokenizer, gen_ids)

    # ── Visualise denoising trajectory ────────────────────────────────────
    @torch.no_grad()
    def visualise_denoising(
        self,
        user_message: str,
        record_every: int = 8,
    ) -> List[str]:
        """
        Record intermediate denoising states (every `record_every` steps).
        Returns a list of partially-decoded strings showing how the response
        evolves from fully masked to complete.
        """
        self.model.eval()
        prompt_ids = format_prompt(self.tokenizer, user_message).to(self.device)
        B, P  = prompt_ids.shape
        L     = self.cfg.gen_length

        input_ids  = torch.cat([prompt_ids,
                                 torch.full((B, L), self.mask_id, dtype=torch.long, device=self.device)],
                                dim=1)
        mask_index = torch.zeros(B, P + L, dtype=torch.bool, device=self.device)
        mask_index[:, P:] = True
        schedule   = get_transfer_schedule(mask_index, self.cfg.num_steps)

        snapshots = []
        MASK_STR  = '▓'

        def snapshot_str(ids):
            tokens = ids[0, P:].tolist()
            parts  = []
            for t in tokens:
                if t == self.mask_id:
                    parts.append(MASK_STR)
                else:
                    parts.append(self.tokenizer.decode([t]))
            return ''.join(parts)

        snapshots.append(f'[step  0] {snapshot_str(input_ids)}')

        for step in range(self.cfg.num_steps):
            logits     = self.model(input_ids).logits
            sampled    = logits.argmax(dim=-1)
            confidence = logits.softmax(dim=-1).max(dim=-1).values
            confidence[~mask_index] = -torch.inf

            k = schedule[:, step]
            for b in range(B):
                ki = k[b].item()
                if ki == 0: continue
                masked_pos  = mask_index[b].nonzero(as_tuple=True)[0]
                if len(masked_pos) == 0: continue
                conf_masked = confidence[b][masked_pos]
                topk_idx    = conf_masked.topk(min(ki, len(masked_pos))).indices
                reveal_pos  = masked_pos[topk_idx]
                input_ids[b, reveal_pos]  = sampled[b, reveal_pos]
                mask_index[b, reveal_pos] = False

            if (step + 1) % record_every == 0 or step == self.cfg.num_steps - 1:
                snapshots.append(f'[step {step+1:3d}] {snapshot_str(input_ids)}')

        return snapshots


print('IterativeRefinementWrapper defined.')

## 10. Training: Noise-Injection Fine-Tuning

We run a short demonstration training loop on our 5-example synthetic dataset.  
In practice, replace `DEMO_DATA` with real instruction-following data (e.g. Tulu, SmolTalk, Alpaca).

In [ ]:
cfg_train = DiffusionConfig(
    # Inference
    num_steps        = 64,
    gen_length       = 128,
    temperature      = 0.0,
    remask_strategy  = 'confidence',
    # Training
    mask_rate_min    = 0.15,   # never mask fewer than 15% — encourages recovery
    mask_rate_max    = 1.00,   # allow full masking (pure denoising)
    label_smoothing  = 0.0,
)

wrapper = IterativeRefinementWrapper(model, tokenizer, cfg_train)

print('Starting noise-injection fine-tuning ...')
print('(Demo: 3 epochs on 5 synthetic examples)\n')

losses = wrapper.noise_injection_train(
    dataloader  = dataloader,
    num_epochs  = 3,
    lr          = 2e-5,
    grad_clip   = 1.0,
    warmup_steps= 5,
    log_every   = 1,
)
print(f'\nTraining complete. Final loss: {losses[-1]:.4f}')

## 11. Loss Curve

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 3))
    plt.plot(losses, lw=1.5, color='steelblue', label='MDLM SFT Loss')
    plt.xlabel('Training Step')
    plt.ylabel('Cross-Entropy Loss')
    plt.title('Noise-Injection Training Loss')
    plt.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not installed. Loss values:')
    print(losses)

## 12. Inference: LLaDA Denoising Generation

In [ ]:
# Try a few prompts
test_prompts = [
    'What is a language model?',
    'How does masked diffusion generate text step by step?',
    'What makes ModernBERT different from the original BERT?',
]

for prompt in test_prompts:
    print(f'\n❓ {prompt}')
    resp = wrapper.generate(prompt, verbose=False)
    print(f'💬 {resp}')
    print('─' * 60)

## 13. Visualise Denoising Trajectory

Watch how the response token sequence evolves from `▓▓▓▓...` (all masked) to the final text.

In [ ]:
VIZ_PROMPT = 'Explain the forward diffusion process for text.'
print(f'Visualising denoising for: "{VIZ_PROMPT}"\n')

snapshots = wrapper.visualise_denoising(VIZ_PROMPT, record_every=8)

for snap in snapshots:
    print(snap[:120])  # truncate long lines

## 14. Ablation: Remask Strategy Comparison

LLaDA supports two remasking strategies. Let's compare them qualitatively.

In [ ]:
ABLATION_PROMPT = 'Describe the difference between masked diffusion and autoregressive generation.'

for strategy in ['confidence', 'random']:
    cfg_ab = DiffusionConfig(
        num_steps       = 32,
        gen_length      = 96,
        temperature     = 0.0,
        remask_strategy = strategy,
    )
    wrapper.cfg = cfg_ab
    resp = wrapper.generate(ABLATION_PROMPT)
    print(f'\n[strategy={strategy}]')
    print(resp[:300])
    print('─' * 60)

# Restore original config
wrapper.cfg = cfg_train

## 15. Ablation: Denoising Steps vs Quality

More steps → finer-grained unmasking → generally higher quality (at increased compute cost).

In [ ]:
import time

STEPS_PROMPT = 'What are the key advantages of diffusion language models?'
step_counts  = [8, 16, 32, 64]

print(f'Prompt: "{STEPS_PROMPT}"\n')
for n_steps in step_counts:
    wrapper.cfg = DiffusionConfig(
        num_steps=n_steps, gen_length=96, temperature=0.0, remask_strategy='confidence'
    )
    t0 = time.time()
    resp = wrapper.generate(STEPS_PROMPT)
    elapsed = time.time() - t0
    print(f'[steps={n_steps:3d} | {elapsed:.2f}s]  {resp[:200]}')
    print()

wrapper.cfg = cfg_train

## 16. Ablation: Mask Rate Distribution During Training

Visualise the effect of different `mask_rate_min` values on training signal.

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    configs = [
        (0.0, 1.0, 'Uniform [0, 1] — LLaDA default'),
        (0.15, 1.0, 'Clipped [0.15, 1] — stable signal'),
        (0.5, 1.0, 'High-noise [0.5, 1] — hard denoising'),
    ]
    for ax, (lo, hi, title) in zip(axes, configs):
        samples = np.random.uniform(lo, hi, 10000)
        ax.hist(samples, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
        ax.set_title(title, fontsize=9)
        ax.set_xlabel('Masking rate t')
        ax.set_ylabel('Count')
    plt.suptitle('Noise-Injection Masking Rate Distributions', fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not installed — skipping plot.')

## 17. Save & Load Fine-Tuned Model

In [ ]:
import os

SAVE_PATH = './modernbert_diffusion_finetuned'

# Save
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'Model saved to {SAVE_PATH}')

# Reload
loaded_model = AutoModelForMaskedLM.from_pretrained(
    SAVE_PATH,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if DEVICE.type == 'cuda' else torch.float32,
).to(DEVICE)
loaded_model.eval()
print('Model reloaded successfully.')

# Sanity check
wrapper2 = IterativeRefinementWrapper(loaded_model, tokenizer, cfg_train)
print(wrapper2.generate('What is masked diffusion?'))

## 18. Extending to Real Instruction-Tuning Datasets

Below is a drop-in recipe to use a HuggingFace dataset (e.g. SmolTalk / Tulu 3) instead of the synthetic examples.

This is the exact setup used in dLLM's BERT-Chat experiments.

In [ ]:
# ── NOT RUN BY DEFAULT — uncomment to use real data ────────────────────────
#
# from datasets import load_dataset
#
# # SmolTalk (≈400k instruction pairs, Apache 2.0)
# raw = load_dataset('HuggingFaceTB/smoltalk', 'smol-smoltalk', split='train')
#
# def hf_to_examples(ds, n=5000):
#     """Convert SmolTalk HF dataset to our list-of-dicts format."""
#     examples = []
#     for row in ds.select(range(min(n, len(ds)))):
#         msgs = row['messages']
#         # Find last user / assistant turn
#         user_msg  = next((m['content'] for m in reversed(msgs) if m['role'] == 'user'), '')
#         asst_msg  = next((m['content'] for m in reversed(msgs) if m['role'] == 'assistant'), '')
#         if user_msg and asst_msg:
#             examples.append({'prompt': user_msg, 'response': asst_msg})
#     return examples
#
# examples = hf_to_examples(raw, n=10000)
# full_dataset    = DiffusionSFTDataset(examples, tokenizer, max_length=512)
# full_dataloader = DataLoader(full_dataset, batch_size=8, shuffle=True, num_workers=2)
#
# wrapper.noise_injection_train(
#     dataloader   = full_dataloader,
#     num_epochs   = 1,
#     lr           = 2e-5,
#     warmup_steps = 100,
# )

print('(Real-data recipe is commented out — see cell for details.)')

## 19. Summary & Design Notes

### What we built

```
IterativeRefinementWrapper
├── noise_injection_train()    ← MDLM SFT: variable-rate masking + NLL loss on masks
└── generate()                 ← LLaDA denoising: full mask → iterative unmask
    ├── llada_generate()           Confidence-guided token revelation
    └── visualise_denoising()      Step-by-step snapshot of unmasking
```

### Key design decisions

| Decision | Rationale |
|---|---|
| **Response-only masking** | Standard SFT convention from LLaDA §3 / dLLM; lets model see full prompt |
| **Variable t ~ U[min, max]** | Forces model to handle all noise levels, crucial for high-quality generation at all step counts |
| **Confidence remasking** | Empirically outperforms random remasking (LLaDA §4.2) |
| **No architectural change** | ModernBERT's bidirectional attention already supports denoising — only SFT needed (dLLM §4.1) |
| **Gumbel temperature** | Controls generation stochasticity; 0 = greedy, >0 = diverse |

### Recommended next steps

1. **Scale data** — replace synthetic 5-example set with SmolTalk / Tulu 3 (≥10k examples)
2. **LoRA fine-tuning** — add `peft` LoRA adapters to reduce trainable parameters by ~100×
3. **Running Confidence Remasking (RCR)** — track per-position *maximum* confidence across steps (MDPO paper) instead of single-step confidence
4. **Fast-dLLM decoding** — add KV caching + confidence-threshold early stopping for 2–4× speedup
5. **diffu-GRPO** — reinforce denoising with reward signals for reasoning tasks (dLLM 2026/04 release)

### References
- Nie et al. (2025). *LLaDA: Large Language Diffusion with mAsking*. arXiv:2502.09992
- Zhou et al. (2026). *dLLM: Simple Diffusion Language Modeling*. arXiv:2602.22661
- [`dllm-hub/ModernBERT-base-chat-v0.1`](https://huggingface.co/dllm-hub/ModernBERT-base-chat-v0.1)